# 16 · Crop quality and exemplar count: LVIS, THINGS and ImageNet against BV and VEDB

**Question.** Notebooks 12 and 14 found BV and VEDB cluster CDI domains more than LVIS box crops (AUC +0.10 CLIP,
+0.06 DINOv3). LVIS labels are human-drawn and correct, but LVIS boxes every instance, however small: the median
box short side is 24 px, while BV and VEDB crops passed YOLOE and a CLIP image–label filter. Is the head-cam
advantage about viewpoint, or about which object images enter each category mean? And do THINGS and ImageNet
differ from BV only because they average fewer, or more, exemplars?

**Two analyses**
1. **Crop size and CLIP filter** (LVIS-exclusive 104 categories). A short-side floor applied to every crop dataset
   (none, ≥ 32, ≥ 64 px) × the LVIS CLIP filter: cosine of the L2-normalized crop embedding to
   "a photo of a {word}" ≥ 0.27, the filter BV and VEDB crops already passed (checked: recomputing BV's stored
   `clip_sim` this way agrees to < 0.001). A category stays when every crop dataset keeps ≥ `MIN_N` crops; the
   unmatched AUC on the same categories is reported beside it.
2. **Exemplar count.** Every dataset reduced to `K` exemplars per category (THINGS has 12–24), `N_DRAW` draws, on
   three category sets: valid129 (BV, VEDB, THINGS), LVIS-exclusive 104 (+ LVIS, all crops and ≥ 32 px +
   CLIP filter) and ImageNet-exclusive 75 (+ ImageNet).

**Decisions**
- Same crops and conventions as notebooks 12, 14 and 15: 2025.2 packed BV shards (subject 00270001 excluded) and
  VEDB shards, CLIP ≥ 0.27, crops L2-normalized, category means z-scored across categories within each condition,
  AUC on cosine-distance RDMs, within-domain category bootstrap (self-pairs dropped).
- Detector matching (YOLOE on COCO images) needs a GPU and is not run here.
- The CLIP filter is partly circular for CLIP (it keeps crops CLIP finds typical), as it is for BV and VEDB; the
  size floors alone use no model.

**Outputs:** `outputs/matched_controls/results/` and `figures/`. Caches in `outputs/matched_controls/cache/`
(git-ignored; ~30 min to rebuild, mostly reading 604k LVIS crop files).

In [ ]:
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

# Manuscript figure style (same as analysis/qwen-open-vocab/fig_style.py): Lato, bold axis labels and legends.
import matplotlib.axes, matplotlib.axis, matplotlib.figure
from matplotlib import font_manager
for _f in (Path.cwd().resolve().parents[1] / "talk-figures" / "unr-2026" / "fonts").glob("*.ttf"):
    font_manager.fontManager.addfont(str(_f))
plt.rcParams.update({
    "font.family": ["Lato", "DejaVu Sans", "sans-serif"], "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 11, "axes.labelweight": "bold", "figure.titlesize": 12, "figure.titleweight": "bold",
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#444444", "axes.labelcolor": "#222222",
    "text.color": "#222222", "xtick.color": "#444444", "ytick.color": "#444444"})
def _bold_legend(orig):
    if getattr(orig, "_bold", False):
        return orig
    def legend(self, *args, **kw):
        prop = dict(kw.pop("prop", {}) or {}); prop.setdefault("weight", "bold")
        prop.setdefault("size", kw.pop("fontsize", plt.rcParams["legend.fontsize"]))
        return orig(self, *args, prop=prop, **kw)
    legend._bold = True
    return legend
matplotlib.axes.Axes.legend = _bold_legend(matplotlib.axes.Axes.legend)
matplotlib.figure.Figure.legend = _bold_legend(matplotlib.figure.Figure.legend)
_tick_init = matplotlib.axis.Tick.__init__
if not getattr(_tick_init, "_bold", False):
    def _bold_tick(self, *a, **k):                       # bold tick labels (no rcParam for tick label weight)
        _tick_init(self, *a, **k); self.label1.set_fontweight("bold"); self.label2.set_fontweight("bold")
    _bold_tick._bold = True; matplotlib.axis.Tick.__init__ = _bold_tick
    _grid = matplotlib.axes.Axes.grid                    # no grid lines in any figure
    matplotlib.axes.Axes.grid = lambda self, visible=None, which="major", axis="both", **kw: _grid(self, False, which=which, axis=axis)
plt.rcParams["axes.grid"] = False
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [ ]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

MANUSCRIPT_SCRIPTS = MANUSCRIPT_DIR / "scripts"
PACKED = {"bv": Path("/ccn2a/dataset/babyview/2025.2/outputs/image_embeddings/babyview_full"),
          "vedb": Path("/ccn2a/dataset/babyview/related_data/vedb_1612/outputs/image_embeddings/vedb_full_adults")}
LVIS_EMB = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")
IMAGENET_EMB = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
THINGS_EMB = {"clip": Path("/ccn2/dataset/babyview/outputs_20250312/things_image_embeddings/clip_embeddings"),
              "dinov3": Path("/ccn2/dataset/babyview/outputs_20250312/image_embeddings/things_bv_overlapping_categories_corrected/facebook_dinov3-vitb16-pretrain-lvd1689m")}
GEO_CACHE = OUT / "crop_geometry" / "cache"        # notebook 15 crop geometry (short side per crop)
LVIS_SET, IMAGENET_SET = OUT / "exemplar_sets" / "lvis_exclusive104", OUT / "exemplar_sets" / "imagenet_exclusive75"

MC = OUT / "matched_controls"
MC_CACHE, MC_RESULTS, MC_FIGURES = MC / "cache", MC / "results", MC / "figures"
for d in (MC_CACHE, MC_RESULTS, MC_FIGURES):
    d.mkdir(parents=True, exist_ok=True)

## Parameters

In [ ]:
MIN_SIM = 0.27                      # CLIP image–label threshold (manuscript); also the LVIS filter
EXCLUDE_BV_SUBJECTS = ("00270001",)
SIZE_EDGES = np.array([16, 32, 64, 128])   # short-side bins: <16, 16-31, 32-63, 64-127, >=128 px
LOWER = np.r_[0, SIZE_EDGES]               # lower edge of each bin
FLOORS = (0, 32, 64)
MIN_N = 10            # crops per category per crop dataset after filtering (analysis 1)
N_BOOT = 1000         # category bootstrap draws (analysis 1)
K = 12                # exemplars per category (analysis 2)
N_DRAW = 500          # exemplar draws (analysis 2)
N_SAMPLE = 300        # cached exemplars per category for BV, VEDB, ImageNet
SEED = 16
RECOMPUTE_CACHE = False

MODELS = ("clip", "dinov3")
MLABEL = {"clip": "CLIP", "dinov3": "DINOv3"}
LABEL = {"bv": "BabyView", "vedb": "VEDB", "things": "THINGS", "imagenet": "ImageNet", "lvis": "LVIS crops",
         "lvis_ge32_clipfilter": "LVIS ≥32 px + CLIP filter"}
COLOR = {"bv": "#0D47A1", "vedb": "#E07B39", "things": "#90CAF9", "imagenet": "#B9A6E0", "lvis": "#4DB38A",
         "lvis_ge32_clipfilter": "#A8DCC0"}   # same as notebooks 07-15
VARIANTS = [("none", 0, False), ("ge32", 32, False), ("ge64", 64, False),
            ("clipfilter", 0, True), ("ge32_clipfilter", 32, True), ("ge64_clipfilter", 64, True)]
VLABEL = {"none": "unmatched", "ge32": "≥32 px", "ge64": "≥64 px", "clipfilter": "CLIP filter",
          "ge32_clipfilter": "≥32 px +\nCLIP filter", "ge64_clipfilter": "≥64 px +\nCLIP filter"}

## Shared helpers

In [ ]:
import sys, time
sys.path.insert(0, str(MANUSCRIPT_SCRIPTS))
from packed_crop_embeddings import (_row_mask, _vectors_for_mask, iter_shard_paths, l2_normalize_rows,
                                    subject_id_from_video_id)


def load_cdi_semantic_map() -> dict[str, str]:
    """``uni_lemma`` → CDI semantic domain from ``data/cdi_words.csv`` (same as notebooks 09-15)."""
    df = pd.read_csv(DATA_DIR / "cdi_words.csv", usecols=["uni_lemma", "category"])
    df["uni_lemma"] = df["uni_lemma"].astype(str).str.strip().str.lower()
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    return df.drop_duplicates(subset=["uni_lemma"], keep="first").set_index("uni_lemma")["category"].to_dict()


CATS = sorted(l.strip().lower() for l in VALID129_TXT.read_text().splitlines() if l.strip())
CIX = {c: i for i, c in enumerate(CATS)}
CDI = load_cdi_semantic_map()
NB = len(SIZE_EDGES) + 1


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def zscore_rows(X, eps=1e-10):
    return (X - X.mean(axis=0)) / (X.std(axis=0) + eps)


def rdm(M):
    Z = zscore_rows(M); Z = Z / np.linalg.norm(Z, axis=1, keepdims=True); return 1 - Z @ Z.T


def auc(R, sem, idx=None):
    """P(between-domain distance > within-domain distance) over category pairs; ties count half."""
    idx = np.arange(len(sem)) if idx is None else idx
    s = sem[idx]; r = R[np.ix_(idx, idx)]; iu = np.triu_indices(len(idx), 1); keep = idx[iu[0]] != idx[iu[1]]
    v = r[iu][keep]; same = (s[iu[0]] == s[iu[1]])[keep]; w, b = v[same], v[~same]
    rk = rankdata(np.concatenate([w, b]))
    return (rk[len(w):].sum() - len(b) * (len(b) + 1) / 2) / (len(w) * len(b))


def domain_bootstrap(sem, n, seed):
    rng = np.random.default_rng(seed); doms = sorted(set(sem))
    return [np.concatenate([rng.choice(np.flatnonzero(sem == d), (sem == d).sum()) for d in doms]) for _ in range(n)]


def ci_p(x):
    lo, hi = np.percentile(x, [2.5, 97.5]); return lo, hi, float(min(1, 2 * min((x <= 0).mean(), (x >= 0).mean())))


def cat_means(cat, X, n_cat):
    S = np.zeros((n_cat, X.shape[1])); np.add.at(S, cat, X.astype(np.float64)); n = np.bincount(cat, minlength=n_cat)
    return S / np.maximum(n, 1)[:, None], n


def unit(v):
    return v / max(np.linalg.norm(v), 1e-12)

## Stage A · caches

- **BV, VEDB:** one pass over the packed shards. Per (category, short-side bin) sums of L2-normalized vectors and
  counts, both models; plus a fixed random sample of up to `N_SAMPLE` crops per category with both embeddings.
  Short sides come from notebook 15's geometry table; every shard crop must find its row.
- **LVIS:** every box crop in the 104 categories (both models; crops with no embedding file are skipped and
  counted), its short side and its CLIP similarity to "a photo of a {word}".
- **ImageNet:** up to `N_SAMPLE` images per CDI word (imagenet_exclusive75 class pooling).
- **THINGS:** all images of the valid129 categories.

In [ ]:
def build_packed(ds):
    g = pd.read_pickle(GEO_CACHE / f"{ds}_crop_geometry.pkl")
    g["bin"] = np.searchsorted(SIZE_EDGES, g.short_px.to_numpy(), side="right").astype(np.int8)
    rng = np.random.default_rng(SEED); pick = set()
    for c, d in g[g.has_dino].groupby("cat"):
        pick.update(rng.choice(d.image_id.to_numpy(), min(N_SAMPLE, len(d)), replace=False))
    look = g.set_index("image_id")[["bin", "short_px"]]
    sums = {m: np.zeros((len(CATS) * NB, 512 if m == "clip" else 768)) for m in MODELS}
    cnt = {m: np.zeros(len(CATS) * NB, np.int64) for m in MODELS}
    ex = {m: {} for m in MODELS}; paths = list(iter_shard_paths(PACKED[ds])); n_rows = 0
    for j, p in enumerate(paths):
        if ds == "bv" and subject_id_from_video_id(p.stem) in EXCLUDE_BV_SUBJECTS:
            continue
        with np.load(p, allow_pickle=True) as z:
            sh = {k: z[k] for k in z.files}
        for m in MODELS:
            mk = _row_mask(sh, categories=set(CATS), min_sim=MIN_SIM, passed_only=False, model=m)
            if not mk.any():
                continue
            ids = sh["image_id"][mk].astype(str); b = look["bin"].reindex(ids).to_numpy()
            assert not np.isnan(b.astype(float)).any(), f"{p.stem}: crops missing from the geometry table"
            cat = np.array([CIX[str(x).strip().lower()] for x in sh["class_name"][mk]])
            X = l2_normalize_rows(_vectors_for_mask(sh, mk, m)); key = cat * NB + b.astype(np.int64)
            np.add.at(cnt[m], key, 1)
            uk, inv = np.unique(key, return_inverse=True)
            Sm = np.zeros((len(uk), X.shape[1])); np.add.at(Sm, inv, X); sums[m][uk] += Sm
            n_rows += int(mk.sum()) if m == "clip" else 0
            for i in np.flatnonzero([x in pick for x in ids]):
                ex[m][ids[i]] = (cat[i], X[i].astype(np.float32))
        if j % 500 == 0:
            log(f"{ds}: {j}/{len(paths)} shards")
    assert n_rows == len(g), f"{ds}: {n_rows} shard crops vs {len(g)} geometry crops"
    np.savez(MC_CACHE / f"{ds}_size_sums.npz", cats=np.array(CATS), size_edges=SIZE_EDGES,
             **{f"{m}_sum": sums[m].reshape(len(CATS), NB, -1).astype(np.float32) for m in MODELS},
             **{f"{m}_n": cnt[m].reshape(len(CATS), NB) for m in MODELS})
    ids = sorted(set(ex["clip"]) & set(ex["dinov3"]))
    np.savez(MC_CACHE / f"{ds}_exemplars.npz", image_id=np.array(ids), cat=np.array([ex["clip"][i][0] for i in ids]),
             short_px=look["short_px"].reindex(ids).to_numpy(), **{f"{m}_X": np.stack([ex[m][i][1] for i in ids]) for m in MODELS})
    log(f"{ds}: {n_rows:,} crops, {len(ids):,} cached exemplars")


def clip_text(words):
    import torch
    from transformers import CLIPModel, CLIPTokenizer
    torch.set_num_threads(1); name = "openai/clip-vit-base-patch32"
    tok = CLIPTokenizer.from_pretrained(name); model = CLIPModel.from_pretrained(name).eval()
    with torch.no_grad():
        t = model.get_text_features(**tok([f"a photo of a {w}" for w in words], padding=True, return_tensors="pt"))
    t = t if isinstance(t, torch.Tensor) else t.pooler_output
    return l2_normalize_rows(t.numpy().astype(np.float64))


def build_lvis():
    cats = [l.strip().lower() for l in open(LVIS_SET / "categories.txt") if l.strip()]
    cc = pd.read_csv(LVIS_SET / "lvis_category_classes.csv")
    m = pd.read_csv(OUT / "mapping" / "lvis_crop_metadata.csv.gz", usecols=["crop_id", "category_id"])
    b = pd.read_csv(OUT / "mapping" / "lvis_crop_manifest.csv").rename(columns={"image_id": "crop_id"})
    d = (m.merge(b, on="crop_id").merge(cc[["cdi_category", "lvis_category_id"]], left_on="category_id", right_on="lvis_category_id"))
    d = d[d.cdi_category.isin(cats)].sort_values("crop_id").reset_index(drop=True)
    d["short_px"] = np.minimum(d.xmax - d.xmin, d.ymax - d.ymin)
    dirs = {"clip": LVIS_EMB / "openai_clip-vit-base-patch32", "dinov3": LVIS_EMB / "facebook_dinov3-vitb16-pretrain-lvd1689m"}
    X, ok = {}, np.ones(len(d), bool)
    for mm, dr in dirs.items():
        out = np.zeros((len(d), 512 if mm == "clip" else 768), np.float16)
        for i, c in enumerate(d.crop_id):
            f = dr / f"{c}.npy"
            if not ok[i] or not f.exists():
                ok[i] = False; continue
            out[i] = unit(np.load(f).astype(np.float64))
            if i % 100000 == 0:
                log(f"lvis {mm}: {i:,}/{len(d):,}")
        X[mm] = out
    log(f"lvis: {(~ok).sum():,} crops without an embedding file skipped")
    d = d[ok].reset_index(drop=True); X = {mm: v[ok] for mm, v in X.items()}
    T = clip_text(cats); ci = d.cdi_category.map({c: i for i, c in enumerate(cats)}).to_numpy()
    sim = np.einsum("ij,ij->i", X["clip"].astype(np.float64), T[ci])
    np.savez(MC_CACHE / "lvis_crops.npz", cats=np.array(cats), crop_id=d.crop_id.to_numpy().astype(str), cat=ci,
             short_px=d.short_px.to_numpy(), clip_sim=sim, clip_X=X["clip"], dinov3_X=X["dinov3"])


def build_imagenet():
    import os, re
    cats = [l.strip().lower() for l in open(IMAGENET_SET / "categories.txt") if l.strip()]
    cc = pd.read_csv(IMAGENET_SET / "imagenet_category_classes.csv"); w2c = dict(zip(cc.wnid, cc.cdi_category))
    dirs = {"clip": IMAGENET_EMB / "openai_clip-vit-base-patch32", "dinov3": IMAGENET_EMB / "facebook_dinov3-vitb16-pretrain-lvd1689m"}
    pat = re.compile(r"(n\d{8})"); by = {}
    for n in os.listdir(dirs["clip"]):
        if n.endswith(".npy") and (w := pat.search(n).group(1)) in w2c:
            by.setdefault(w2c[w], []).append(n)
    rng = np.random.default_rng(SEED); rows = []
    for c in cats:
        f = sorted(by.get(c, []))
        rows += [(c, n) for n in rng.choice(f, min(N_SAMPLE, len(f)), replace=False)]
    X = {mm: np.stack([unit(np.load(dr / n).astype(np.float64).ravel()) for _, n in rows]).astype(np.float32) for mm, dr in dirs.items()}
    np.savez(MC_CACHE / "imagenet_exemplars.npz", cats=np.array(cats), cat=np.array([cats.index(c) for c, _ in rows]),
             fname=np.array([n for _, n in rows]), **{f"{mm}_X": X[mm] for mm in X})


def build_things():
    rows = {m: [] for m in MODELS}; cat = {m: [] for m in MODELS}
    for c in CATS:
        for m in MODELS:
            sub = [p for p in THINGS_EMB[m].iterdir() if p.is_dir() and p.name.lower() == c]
            for f in (sorted(sub[0].rglob("*.npy")) if sub else []):
                rows[m].append(unit(np.load(f).astype(np.float64).ravel()).astype(np.float32)); cat[m].append(CIX[c])
    assert cat["clip"] == cat["dinov3"], "THINGS images differ between models"
    np.savez(MC_CACHE / "things_exemplars.npz", cats=np.array(CATS), cat=np.array(cat["clip"]),
             **{f"{m}_X": np.stack(rows[m]) for m in MODELS})


BUILD = {"things": build_things, "imagenet": build_imagenet, "lvis": build_lvis,
         "vedb": lambda: build_packed("vedb"), "bv": lambda: build_packed("bv")}
FILE = {"things": "things_exemplars.npz", "imagenet": "imagenet_exemplars.npz", "lvis": "lvis_crops.npz",
        "vedb": "vedb_size_sums.npz", "bv": "bv_size_sums.npz"}
for k, f in BUILD.items():
    if RECOMPUTE_CACHE or not (MC_CACHE / FILE[k]).exists():
        log("building", k); f()

lv = dict(np.load(MC_CACHE / "lvis_crops.npz")); LCATS = list(lv["cats"])
packed = {ds: dict(np.load(MC_CACHE / f"{ds}_size_sums.npz")) for ds in ("bv", "vedb")}
ex = {ds: dict(np.load(MC_CACHE / f"{ds}_exemplars.npz")) for ds in ("bv", "vedb", "things", "imagenet")}
ICATS = list(ex["imagenet"]["cats"])
run = json.load(open(LVIS_SET / "lvis_exemplar_embedding_run.json"))
n_ref = run["n_exemplars_per_category"]; n_now = np.bincount(lv["cat"], minlength=len(LCATS))
assert all(n_ref[c] == n for c, n in zip(LCATS, n_now)), "LVIS crop counts differ from notebook 03's run"
for ds in ("bv", "vedb"):
    print(f"{ds}: {packed[ds]['clip_n'].sum():,} crops (CLIP), {len(ex[ds]['cat']):,} cached exemplars")
print(f"LVIS: {len(lv['cat']):,} crops (= notebook 03's counts), CLIP filter keeps {(lv['clip_sim'] >= MIN_SIM).mean():.1%}")
print(f"THINGS: {len(ex['things']['cat']):,} images; ImageNet: {len(ex['imagenet']['cat']):,} cached images")

### Crop sizes

LVIS boxes every instance, so most of its crops are far smaller than BV's and VEDB's.

In [ ]:
g15 = {ds: pd.read_pickle(GEO_CACHE / f"{ds}_crop_geometry.pkl")["short_px"].to_numpy() for ds in ("bv", "vedb")}
size_tab = pd.DataFrame({LABEL[k]: {"crops": len(v), "median short side (px)": np.median(v), "share < 32 px": (v < 32).mean(),
                                    "share ≥ 64 px": (v >= 64).mean()}
                         for k, v in {**g15, "lvis": lv["short_px"]}.items()}).T
size_tab.to_csv(MC_RESULTS / "crop_size_by_dataset.csv")
display(size_tab.round(3))


def fig_sizes():
    fig, ax = plt.subplots(figsize=(7.5, 3.4), layout="constrained")
    bins = np.logspace(0, np.log10(2048), 60)
    for k, v in {**g15, "lvis": lv["short_px"]}.items():
        ax.hist(v, bins=bins, density=True, histtype="step", lw=2, color=COLOR[k], label=f"{LABEL[k]} (median {np.median(v):.0f} px)")
    for t in (32, 64):
        ax.axvline(t, color="0.5", ls="--", lw=1)
    ax.set_xscale("log"); ax.set_xlabel("box short side (px)"); ax.set_ylabel("density"); ax.legend(frameon=False, fontsize=9)
    ax.set_title("Crop size: LVIS boxes every instance; BV and VEDB crops passed a detector and CLIP filter", fontsize=10)
    out = MC_FIGURES / "crop_size_by_dataset.png"; fig.savefig(out, dpi=200); fig.savefig(out.with_suffix(".pdf")); plt.close(fig); return out

## Analysis 1 · crop size and CLIP filter (LVIS-exclusive 104)

The `none` rows must reproduce notebook 12's LVIS comparison (BV 0.771 / 0.776, LVIS 0.674 / 0.714).

In [ ]:
def packed_means(ds, m, cats, floor):
    z = packed[ds]; idx = [CATS.index(c) for c in cats]; keep = LOWER >= floor
    S = z[f"{m}_sum"][idx][:, keep].sum(1).astype(np.float64); n = z[f"{m}_n"][idx][:, keep].sum(1)
    return S / np.maximum(n, 1)[:, None], n


def lvis_means(m, floor, clip_filter):
    mk = lv["short_px"] >= floor
    if clip_filter:
        mk &= lv["clip_sim"] >= MIN_SIM
    return cat_means(lv["cat"][mk], lv[f"{m}_X"][mk], len(LCATS))


def things_means(m, cats):
    z = ex["things"]; M, n = cat_means(z["cat"], z[f"{m}_X"], len(CATS)); idx = [CATS.index(c) for c in cats]
    return M[idx], n[idx]


PAIRS = [("bv", "lvis"), ("vedb", "lvis"), ("things", "lvis"), ("bv", "vedb"), ("bv", "things")]
rows, diffs = [], []
for m in MODELS:
    base = {"bv": packed_means("bv", m, LCATS, 0), "vedb": packed_means("vedb", m, LCATS, 0),
            "lvis": lvis_means(m, 0, False), "things": things_means(m, LCATS)}
    for name, floor, cf in VARIANTS:
        cur = {"bv": packed_means("bv", m, LCATS, floor), "vedb": packed_means("vedb", m, LCATS, floor),
               "lvis": lvis_means(m, floor, cf), "things": base["things"]}
        ok = np.all([cur[d][1] >= MIN_N for d in ("bv", "vedb", "lvis")], axis=0) & (base["things"][1] > 0)
        cats = [c for c, o in zip(LCATS, ok) if o]; sem = np.array([CDI.get(c, "other") for c in cats])
        R = {d: rdm(cur[d][0][ok]) for d in cur}; R0 = {d: rdm(base[d][0][ok]) for d in base}
        boots = domain_bootstrap(sem, N_BOOT, 3)
        B = {d: np.array([auc(R[d], sem, b) for b in boots]) for d in R}
        B0 = {d: np.array([auc(R0[d], sem, b) for b in boots]) for d in R0}
        for d in R:
            a, a0 = auc(R[d], sem), auc(R0[d], sem); lo, hi, _ = ci_p(B[d])
            rows.append(dict(model=m, variant=name, floor_px=floor, lvis_clip_filter=cf, dataset=d, n_categories=len(cats),
                             median_crops_per_category=int(np.median(cur[d][1][ok])), auc=a, auc_ci_lo=lo, auc_ci_hi=hi,
                             auc_unmatched_same_categories=a0, change_vs_unmatched=a - a0, change_p=ci_p(B[d] - B0[d])[2]))
        for a_, b_ in PAIRS:
            lo, hi, p = ci_p(B[a_] - B[b_])
            diffs.append(dict(model=m, variant=name, pair=f"{a_}_minus_{b_}", n_categories=len(cats),
                              diff=auc(R[a_], sem) - auc(R[b_], sem), ci_lo=lo, ci_hi=hi, p_boot=p,
                              diff_unmatched_same_categories=auc(R0[a_], sem) - auc(R0[b_], sem)))
A = pd.DataFrame(rows); AD = pd.DataFrame(diffs)
A.to_csv(MC_RESULTS / "size_clip_matching_summary.csv", index=False)
AD.to_csv(MC_RESULTS / "size_clip_matching_differences.csv", index=False)
chk = A[A.variant == "none"].set_index(["model", "dataset"]).auc
assert abs(chk["clip", "bv"] - 0.771) < 0.001 and abs(chk["clip", "lvis"] - 0.674) < 0.001, "unmatched AUCs differ from notebook 12"
display(A.pivot_table(index=["model", "variant", "n_categories"], columns="dataset", values="auc", sort=False).round(3))
fmt = lambda r: f"{r['diff']:+.3f} [{r.ci_lo:+.3f}, {r.ci_hi:+.3f}], p = {r.p_boot:.3f}"
AD["text"] = AD.apply(fmt, axis=1)
display(AD[AD.pair.isin(["bv_minus_lvis", "vedb_minus_lvis", "things_minus_lvis"])]
        .pivot_table(index=["model", "variant"], columns="pair", values="text", aggfunc="first", sort=False))

In [ ]:
def fig_matching():
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True, layout="constrained")
    x = np.arange(len(VARIANTS)); off = {"bv": -0.21, "vedb": -0.07, "lvis": 0.07, "things": 0.21}
    DF = pd.read_csv(MC_FIGURES.parent / "results" / "size_clip_matching_differences.csv")   # bootstrap p, BabyView vs each
    star = lambda p: "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "n.s."

    for ax, m in zip(axes, MODELS):
        s = A[A.model == m]
        for d, o in off.items():
            r = s[s.dataset == d].set_index("variant").loc[[v for v, _, _ in VARIANTS]]
            ax.errorbar(x + o, r.auc, yerr=[r.auc - r.auc_ci_lo, r.auc_ci_hi - r.auc], fmt="o", ms=7, color=COLOR[d],
                        mec="0.2", ecolor=COLOR[d], capsize=0, lw=1.5, label=LABEL[d] if d != "lvis" else "LVIS crops")
            if d != "bv":                                # differs from BabyView (category bootstrap p)
                for xi, (v, _, _), hi in zip(x + o, VARIANTS, r.auc_ci_hi):
                    p_ = DF[(DF.model == m) & (DF.variant == v) & (DF.pair == f"bv_minus_{d}")].p_boot
                    if len(p_) and star(p_.iloc[0]) != "n.s.":
                        ax.text(xi, hi + .004, star(p_.iloc[0]), ha="center", va="bottom", fontsize=8, fontweight="bold")
        ax.set_xticks(x); ax.set_xticklabels([VLABEL[v] for v, _, _ in VARIANTS], fontsize=9)
        ax.set_title(MLABEL[m]); ax.axhline(0.5, color="0.85", lw=0.8)
    axes[0].set_ylabel("domain cluster strength (AUC)"); axes[0].legend(frameon=False, fontsize=9, loc="lower right")
    fig.suptitle("Matching LVIS crop quality: size floors on every dataset, CLIP filter on LVIS (104 categories; 95% category bootstrap)\nabove LVIS, VEDB and THINGS: differs from BabyView (* p < .05, ** < .01, *** < .001; no mark: n.s.)", fontsize=10)
    out = MC_FIGURES / "crop_quality_matching_auc.png"; fig.savefig(out, dpi=200); fig.savefig(out.with_suffix(".pdf")); plt.close(fig); return out


def fig_diffs():
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True, layout="constrained")
    x = np.arange(len(VARIANTS))
    for ax, m in zip(axes, MODELS):
        for pair, o, d in [("bv_minus_lvis", -0.1, "bv"), ("vedb_minus_lvis", 0.1, "vedb")]:
            r = AD[(AD.model == m) & (AD.pair == pair)].set_index("variant").loc[[v for v, _, _ in VARIANTS]]
            ax.errorbar(x + o, r["diff"], yerr=[r["diff"] - r.ci_lo, r.ci_hi - r["diff"]], fmt="o", ms=7, color=COLOR[d],
                        mec="0.2", ecolor=COLOR[d], lw=1.5, label=f"{LABEL[d]} − LVIS")
        ax.axhline(0, color="0.3", lw=1); ax.set_xticks(x); ax.set_xticklabels([VLABEL[v] for v, _, _ in VARIANTS], fontsize=9)
        ax.set_title(MLABEL[m])
    axes[0].set_ylabel("AUC difference"); axes[0].legend(frameon=False, fontsize=9)
    fig.suptitle("Head-cam minus LVIS AUC after matching crop quality (95% category bootstrap)", fontsize=10)
    out = MC_FIGURES / "crop_quality_matching_differences.png"; fig.savefig(out, dpi=200); fig.savefig(out.with_suffix(".pdf")); plt.close(fig); return out

## Analysis 2 · exemplar count

Each draw takes `K` exemplars per category from every dataset, averages them, and computes AUC on all categories
(exemplar sampling only; the reported mean and draw interval). Differences add a within-domain category bootstrap
in the same draw (CIs and p). A category stays when every dataset in its set has ≥ `K` exemplars.

In [ ]:
lv_pools = {"lvis": np.ones(len(lv["cat"]), bool), "lvis_ge32_clipfilter": (lv["short_px"] >= 32) & (lv["clip_sim"] >= MIN_SIM)}


def pools_for(ds, cats):
    if ds.startswith("lvis"):
        cat, mk, names = lv["cat"], lv_pools[ds], LCATS; src = lambda m: lv[f"{m}_X"]
    else:
        z = ex[ds]; cat, mk = z["cat"], np.ones(len(z["cat"]), bool); src = lambda m, z=z: z[f"{m}_X"]
        names = ICATS if ds == "imagenet" else CATS
    return [np.flatnonzero(mk & (cat == names.index(c))) if c in names else np.array([], int) for c in cats], src


SETS = {"valid129": (CATS, ["bv", "vedb", "things"]),
        "lvis104": (LCATS, ["bv", "vedb", "things", "lvis", "lvis_ge32_clipfilter"]),
        "imagenet75": (ICATS, ["bv", "vedb", "things", "imagenet"])}
brows, bdiffs = [], []
for set_name, (cats0, dss) in SETS.items():
    P = {ds: pools_for(ds, cats0) for ds in dss}
    ok = np.all([[len(ix) >= K for ix in P[ds][0]] for ds in dss], axis=0)
    cats = [c for c, o in zip(cats0, ok) if o]; sem = np.array([CDI.get(c, "other") for c in cats])
    pools = {ds: [ix for ix, o in zip(P[ds][0], ok) if o] for ds in dss}
    boots = domain_bootstrap(sem, N_DRAW, 7)
    for m in MODELS:
        rng = np.random.default_rng(11); A0 = {ds: np.zeros(N_DRAW) for ds in dss}; AB = {ds: np.zeros(N_DRAW) for ds in dss}
        for r in range(N_DRAW):
            for ds in dss:
                X = P[ds][1](m)
                M = np.stack([X[rng.choice(ix, K, replace=False)].astype(np.float64).mean(0) for ix in pools[ds]])
                R = rdm(M); A0[ds][r] = auc(R, sem); AB[ds][r] = auc(R, sem, boots[r])
        for ds in dss:
            brows.append(dict(set=set_name, model=m, dataset=ds, k=K, n_categories=len(cats), auc_mean=A0[ds].mean(),
                              auc_draw_lo=np.percentile(A0[ds], 2.5), auc_draw_hi=np.percentile(A0[ds], 97.5)))
        for i, a_ in enumerate(dss):
            for b_ in dss[i + 1:]:
                lo, hi, p = ci_p(AB[a_] - AB[b_])
                bdiffs.append(dict(set=set_name, model=m, pair=f"{a_}_minus_{b_}", k=K, n_categories=len(cats),
                                   diff=(A0[a_] - A0[b_]).mean(), ci_lo=lo, ci_hi=hi, p_boot=p))
BS = pd.DataFrame(brows); BD = pd.DataFrame(bdiffs)
BS.to_csv(MC_RESULTS / "exemplar_count_summary.csv", index=False); BD.to_csv(MC_RESULTS / "exemplar_count_differences.csv", index=False)
display(BS.pivot_table(index=["set", "model", "n_categories"], columns="dataset", values="auc_mean", sort=False).round(3))
BD["text"] = BD.apply(fmt, axis=1)
display(BD[BD.pair.str.startswith(("bv_", "vedb_minus_things"))].pivot_table(index=["set", "pair"], columns="model", values="text", aggfunc="first", sort=False))

In [ ]:
def fig_exemplars():
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True, layout="constrained", gridspec_kw={"width_ratios": [3, 5, 4]})
    for ax, (set_name, (_, dss)) in zip(axes, SETS.items()):
        s = BS[BS.set == set_name]; x = np.arange(len(dss))
        for m, o, mk in [("clip", -0.12, "o"), ("dinov3", 0.12, "s")]:
            r = s[s.model == m].set_index("dataset").loc[dss]
            for i, d in enumerate(dss):
                ax.errorbar(x[i] + o, r.loc[d, "auc_mean"], yerr=[[r.loc[d, "auc_mean"] - r.loc[d, "auc_draw_lo"]], [r.loc[d, "auc_draw_hi"] - r.loc[d, "auc_mean"]]],
                            fmt=mk, ms=7, color=COLOR[d], mec="0.2", ecolor=COLOR[d], lw=1.5,
                            mfc=COLOR[d] if m == "clip" else "white")
        ax.set_xticks(x); ax.set_xticklabels([LABEL[d].replace(" + ", "\n+ ") for d in dss], fontsize=9)
        ax.set_title(f"{set_name} ({s.n_categories.iloc[0]} categories)", fontsize=10)
    axes[0].set_ylabel(f"AUC, {K} exemplars per category")
    axes[0].legend(handles=[plt.Line2D([], [], marker="o", ls="", color="0.4", label="CLIP"),
                            plt.Line2D([], [], marker="s", ls="", color="0.4", mfc="white", label="DINOv3")], frameon=False, fontsize=9)
    fig.suptitle(f"Exemplar count matched: {K} per category in every dataset ({N_DRAW} draws; bars = 95% of draws)", fontsize=10)
    out = MC_FIGURES / "exemplar_count_auc.png"; fig.savefig(out, dpi=200); fig.savefig(out.with_suffix(".pdf")); plt.close(fig); return out


show_images([fig_sizes(), fig_matching(), fig_diffs(), fig_exemplars()], width=950)

### Review checklist

- Do the unmatched rows reproduce notebook 12 (asserted), and do LVIS counts match notebook 03's run (asserted)?
- How many categories survive each floor, and how many crops per category remain (`median_crops_per_category`)?
  LVIS ≥ 64 px + CLIP filter rests on a median of ~350 crops per category.
- Which side moves? If BV and VEDB stay put while LVIS rises, the original gap came from LVIS's small,
  unrecognizable crops, not from what head-cams see.
- The CLIP-filter variants keep crops CLIP finds typical, which can raise CLIP clustering by construction; read the
  size-only floors as the cleaner test, and DINOv3 as the less circular model.
- Exemplar count: THINGS images are curated and sharp; averaging 12 of them is not the same as 12 random head-cam
  crops, so this equalizes the count, not the quality.